# Формирование временных рядов

In [6]:
import polars as pl
import pandas as pd

In [7]:
FILE_PATH = "ШЧ_2_corr.xlsx"

# 1. Определение количества листов
xls = pd.ExcelFile(FILE_PATH, engine="openpyxl")
sheet_names = xls.sheet_names
n_sheets = len(sheet_names)
print(f"Количество листов: {n_sheets}")
print(f"Названия листов: {sheet_names}")
# Объявляем список,
dfs = []
for sheet in sheet_names:    
    df = pl.read_excel(FILE_PATH, sheet_name=sheet)    
    # способ добавить столбец в Polars (pl.lit(sheet) — создание литерала (константы), .alias("Год_листа") — присвоение имени)
    df = df.with_columns(
        pl.lit(sheet).alias("Год_листа")
    )
    dfs.append(df)
    # объединяем все датафреймы в один (диагонально - на случай разных наборов колонок)
result = pl.concat(dfs, how="diagonal_relaxed")


Количество листов: 6
Названия листов: ['2019', '2018', '2017', '2016', '2015', '2014']


Could not determine dtype for column 10, falling back to string
C:\Users\gruni\AppData\Local\Temp\ipykernel_2376\3521003566.py:12: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  df = pl.read_excel(FILE_PATH, sheet_name=sheet)
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 10, falling back to string


In [8]:
# Сделаем копию нашего массива. ВОПРОС: для чего?
df = result.clone()
# Приводим колонку "Дата осмотра" к типу datetime, если она строковая
df = df.with_columns( 
    pl.col("Дата осмотра").str.strptime(pl.Datetime, strict=False, format=None)
    if df.schema["Дата осмотра"] == pl.Utf8
    else pl.col("Дата осмотра")
)

# ===========================================================
# 5.1 Извлечение признаков из даты
# ===========================================================
df = df.with_columns([
    pl.col("Дата осмотра").dt.week().alias("Номер_недели"),      # номер недели в году
    pl.col("Дата осмотра").dt.ordinal_day().alias("Номер_дня"),  # номер дня в году
    pl.col("Дата осмотра").dt.month().alias("Номер_месяца"),     # номер месяца
    pl.col("Дата осмотра").dt.year().alias("Год"),                # год
])

print("\nПример новых признаков даты:")
print(df.select(["Дата осмотра", "Год", "Номер_месяца", "Номер_недели", "Номер_дня"]).head(10))


Пример новых признаков даты:
shape: (10, 5)
┌──────────────┬──────┬──────────────┬──────────────┬───────────┐
│ Дата осмотра ┆ Год  ┆ Номер_месяца ┆ Номер_недели ┆ Номер_дня │
│ ---          ┆ ---  ┆ ---          ┆ ---          ┆ ---       │
│ date         ┆ i32  ┆ i8           ┆ i8           ┆ i16       │
╞══════════════╪══════╪══════════════╪══════════════╪═══════════╡
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4            ┆ 14           ┆ 93        │
│ 2019-04-03   ┆ 2019 ┆ 4      

In [9]:
df[:1]

__UNNAMED__0,Дата осмотра,Осмотр проводил,"Станция, перегон","Объекты, количество отступлений",Классификация (Группа),Классификация (Вид),Примечание,Крайний срок,Статус,Причина,Дата устранения,Год_листа,Номер_недели,Номер_дня,Номер_месяца,Год
i64,date,str,str,str,str,str,str,date,str,str,date,str,i8,i16,i8,i32
0,2019-04-03,"""Сотрудник_0""","""станция_0""","""327""","""группа_0""","""вид_0""","""Стрелка №39 устранить люфт дли…",2019-04-13,"""ЗАКРЫТО""",null,2019-04-11,"""2019""",14,93,4,2019


In [10]:
# Соберем данные для нашего временного ряда
df_agg = (
    df.group_by(["Станция, перегон", "Номер_дня"])
    .agg(pl.col("Дата осмотра").count().alias("Количество инцидентов"))
    .sort(["Станция, перегон", "Номер_дня"])
)
df_agg

"Станция, перегон",Номер_дня,Количество инцидентов
str,i16,u32
"""станция_0""",1,11
"""станция_0""",2,7
"""станция_0""",13,23
"""станция_0""",15,11
"""станция_0""",17,11
…,…,…
"""станция_99""",268,5
"""станция_99""",286,9
"""станция_99""",318,1


In [11]:
# Создаем новый датафрейм
min_day = 1
max_day = 365
pl.DataFrame({"Номер дня": range(min_day, max_day + 1)})

Номер дня
i64
1
2
3
4
5
…
361
362
363


In [12]:
# Выровняем временной ряд
min_day = 1
max_day = 366

df_aligned = (
    df_agg
    .select("Станция, перегон")
    .unique()
    .join(
        pl.DataFrame({"Номер_дня": range(min_day, max_day + 1)}),
        how="cross",
    )
    .join(df_agg, on=["Станция, перегон", "Номер_дня"], how="left")
    .sort(["Станция, перегон", "Номер_дня"])
    .with_columns(
        pl.col("Количество инцидентов")
          .fill_null(0) #заполняем пропущенные значения 0
        # .fill_null(strategy="forward")  # вариант B: последнее известное
        # .interpolate()                  # вариант C: линейная интер
    )
)
df_aligned[:14]

"Станция, перегон",Номер_дня,Количество инцидентов
str,i64,u32
"""станция_0""",1,11
"""станция_0""",2,7
"""станция_0""",3,0
"""станция_0""",4,0
"""станция_0""",5,0
…,…,…
"""станция_0""",10,0
"""станция_0""",11,0
"""станция_0""",12,0


In [13]:
# Добавляем столбец, характеризующий прирост, накопленный итог (важно! расчет ведется в рамках каждой станции)
df_aligned = (
    df_aligned
    .with_columns(
        pl.col("Количество инцидентов")
          .fill_null(0)
          .fill_nan(0)
    )
    .with_columns(
        pl.col("Количество инцидентов")
          .diff()
          .over("Станция, перегон") # это нужно для того чтобы начать расчет с каждой станции с 0
          .alias("Прирост инцидентов")
    )
    .with_columns(
        pl.col("Количество инцидентов")
          .cum_sum()
          .over("Станция, перегон") # это нужно для того чтобы начать расчет с каждой станции с 0
          .alias("Нарощенный итог")
    ))
df_aligned

"Станция, перегон",Номер_дня,Количество инцидентов,Прирост инцидентов,Нарощенный итог
str,i64,u32,i64,u32
"""станция_0""",1,11,null,11
"""станция_0""",2,7,-4,18
"""станция_0""",3,0,-7,18
"""станция_0""",4,0,0,18
"""станция_0""",5,0,0,18
…,…,…,…,…
"""станция_99""",362,0,0,276
"""станция_99""",363,0,0,276
"""станция_99""",364,0,0,276


In [14]:
pip install tslearn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   -------------- ------------------------- 1.0/2.8 MB 5.6 MB/s eta 0:00:01
   ----------------------------- ---------- 2.1/2.8 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 5.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.3 MB 5.0 MB/s eta 0:00:03
   --------- ------------------------------ 2.6/11.3 MB 6.3 MB/s eta 0:00:02
   ------------- -------------------------- 3.9/11.3 MB 6.3 MB/s eta 0:00:02
   ------------------- -------------------- 5.5/11.3 MB 6.3 MB/s eta 0:00:01
   ------------------------ --------------- 6.8/11.3 MB 6.5 MB/s eta 0:00:01
   ------------------------------ --------- 8.7/11.3 MB 6.8 MB/s eta 0:00:01
   ------------------------------------ --- 10.2/11.3 MB 6.8 MB/s eta 0:00:01
   ------------


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\gruni\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [15]:
# Ищем ответ на вопрос: динамика отказов на каких станциях похожа друг на друга?

import numpy as np
import matplotlib.pyplot as plt
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.clustering import TimeSeriesKMeans

C:\Users\gruni\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tslearn\bases\bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


In [16]:
METRICS = [
    "Количество инцидентов",
    "Прирост инцидентов",
    "Нарощенный итог",
]

In [17]:
df_clean = (
    df_aligned
    .sort(["Станция, перегон", "Номер_дня"])
    .fill_null(0)
)

df_grouped = (
    df_aligned
    .sort(["Станция, перегон", "Номер_дня"])
    .fill_null(0)
    .group_by("Станция, перегон", maintain_order=True)
    .agg([
        pl.col("Номер_дня").alias("days"),
        *[pl.col(m).implode().alias(m) for m in METRICS]  # ← implode()
    ])
)
df_grouped


"Станция, перегон",days,Количество инцидентов,Прирост инцидентов,Нарощенный итог
str,list[i64],list[u32],list[i64],list[u32]
"""станция_0""","[1, 2, … 366]","[11, 7, … 0]","[0, -4, … 0]","[11, 18, … 5755]"
"""станция_1""","[1, 2, … 366]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 200]"
"""станция_10""","[1, 2, … 366]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 309]"
"""станция_100""","[1, 2, … 366]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 517]"
"""станция_101""","[1, 2, … 366]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 61]"
…,…,…,…,…
"""станция_95""","[1, 2, … 366]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 142]"
"""станция_96""","[1, 2, … 366]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 507]"
"""станция_97""","[1, 2, … 366]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 130]"


In [18]:
stations_list = df_grouped["Станция, перегон"].to_list()
X_ts = np.array([
    np.column_stack([row[m] for m in METRICS])
    for row in df_grouped.iter_rows(named=True)
])

print(f"Форма массива: {X_ts.shape}")

Форма массива: (118, 366, 3)


In [19]:
# 1. Нормализация (важно для DTW, чтобы амплитуда не доминировала над формой)
scaler = TimeSeriesScalerMeanVariance()
X_scaled = scaler.fit_transform(X_ts)
X_scaled

array([[[-1.49705714e-01,  7.00003859e-04, -1.60004081e+00],
        [-2.76466364e-01, -9.24641461e-02, -1.59609279e+00],
        [-4.98297503e-01, -1.62337259e-01, -1.59609279e+00],
        ...,
        [-2.76466364e-01,  9.38641538e-02,  1.63959502e+00],
        [-4.98297503e-01, -1.62337259e-01,  1.63959502e+00],
        [-4.98297503e-01,  7.00003859e-04,  1.63959502e+00]],

       [[-1.43942865e-01,  0.00000000e+00, -1.53650152e+00],
        [-1.43942865e-01,  0.00000000e+00, -1.53650152e+00],
        [-1.43942865e-01,  0.00000000e+00, -1.53650152e+00],
        ...,
        [-1.43942865e-01,  0.00000000e+00,  1.15827557e+00],
        [-1.43942865e-01,  0.00000000e+00,  1.15827557e+00],
        [-1.43942865e-01,  0.00000000e+00,  1.15827557e+00]],

       [[-2.00365975e-01,  0.00000000e+00, -1.75998197e+00],
        [-2.00365975e-01,  0.00000000e+00, -1.75998197e+00],
        [-2.00365975e-01,  0.00000000e+00, -1.75998197e+00],
        ...,
        [-2.00365975e-01,  0.00000000e+00,

In [ ]:
n_days = X_scaled.shape[1]
sakoe_chiba_radius = max(1, int(n_days * 0.15))

n_clusters = 6
print(f"Запуск DTW кластеризации (окно={sakoe_chiba_radius})...")

# ✅ НОВОЕ: параметры DTW передаются напрямую, а не через dtw_params={}
km_dtw = TimeSeriesKMeans(
    n_clusters=n_clusters,
    metric="dtw",
    max_iter=50,
    random_state=42,
    n_init=5,
) # Гиперпараметр модели.

labels = km_dtw.fit_predict(X_scaled)

# ШАГ 3: Возврат результатов в Polars

df_clusters = pl.DataFrame({
    "Станция, перегон": stations_list,
    "cluster_dtw": labels
})

4

Запуск DTW кластеризации (окно=54)...


4

In [21]:
df_clusters['cluster_dtw'].value_counts()

cluster_dtw,count
i64,u32
5,21
2,22
3,19
0,29
1,16
4,11


In [22]:
df_clusters.filter(pl.col('cluster_dtw')==0)

"Станция, перегон",cluster_dtw
str,i64
"""станция_0""",0
"""станция_1""",0
"""станция_10""",0
"""станция_113""",0
"""станция_114""",0
…,…
"""станция_84""",0
"""станция_85""",0
"""станция_91""",0


In [23]:
# Задание! на график временные ряды станций кластера 0 (или другого). Не используйте GTP, ищите в контенте лекций.
# Посчитайте основые характеристики временных рядов кластера. Подумайте что это за характеристики. 

In [24]:
# Задание 2. По аналогии с кодом выше проведите агрегацию по станциям, по каждой станции нужны характеристики временных рядов, а не многомерные временнные ряды
# целевой массив
                # Признак_1  Признак_2 Признак_3 Признак_4 Признак_5
# Станция_0
# Станция_1
# Станция_2